# Step 1: Data Setup for VQA Quickstart

This notebook loads the **OCR-VQA** dataset from a public S3 bucket and uploads it to Snowflake for batch inference.

OCR-VQA contains images with text (book covers, signs, etc.) and questions about reading that text.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

DB_NAME = "VQA_DEMO_DB"
SCHEMA_NAME = "VQA"

session.sql(f"USE DATABASE {DB_NAME}").collect()
session.sql(f"USE SCHEMA {SCHEMA_NAME}").collect()

print(f"Connected to Snowflake")
print(f"Database: {DB_NAME}")
print(f"Schema: {SCHEMA_NAME}")

## Step 2: Define S3 Source Paths

In [ ]:
S3_BUCKET_URL = 's3://sfquickstarts/sfguide_ml_batch_inference_with_vision_models/'

print(f"S3 source: {S3_BUCKET_URL}")

## Step 3: Create External Stage for S3 Data

In [ ]:
session.sql(f"CREATE OR REPLACE STAGE S3_DATA_STAGE URL='{S3_BUCKET_URL}' DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = FALSE)").collect()

print("Created external stage S3_DATA_STAGE")
session.sql("LS @S3_DATA_STAGE/IMAGES/").show(5)

## Step 4: Copy Images to Internal Stage

In [ ]:
session.sql("CREATE OR REPLACE STAGE IMAGES_STAGE DIRECTORY = (ENABLE = TRUE) ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')").collect()

session.sql("COPY FILES INTO @IMAGES_STAGE/ FROM @S3_DATA_STAGE/IMAGES/").collect()

print("Copied images from S3 to internal stage")
result = session.sql("LS @IMAGES_STAGE").collect()
print(f"Total images in stage: {len(result)}")

## Step 5: Load Q&A Dataset from S3 CSV

In [ ]:
import json
import pandas as pd

session.sql("""
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
    TYPE = 'CSV'
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    ESCAPE_UNENCLOSED_FIELD = NONE
""").collect()

df_vqa = session.sql("""
SELECT 
    $1::INT AS ID,
    $2::INT AS IMAGE_IDX,
    $3::STRING AS IMAGE_PATH,
    $4::STRING AS QUESTION,
    $5::STRING AS ANSWER,
    $6::STRING AS QUESTION_TYPE
FROM @S3_DATA_STAGE/vqa_dataset.csv (FILE_FORMAT => 'CSV_FORMAT')
""").to_pandas()

num_images = df_vqa['IMAGE_IDX'].nunique()
print(f"Created {len(df_vqa)} Q&A records from {num_images} images")
print(f"\nSample questions:")
for i in range(min(5, len(df_vqa))):
    print(f"  Q: {df_vqa.iloc[i]['QUESTION']}")
    print(f"  A: {df_vqa.iloc[i]['ANSWER']}\n")

In [ ]:
df_sf = session.create_dataframe(df_vqa)
df_sf.write.mode("overwrite").save_as_table(f"{DB_NAME}.{SCHEMA_NAME}.VQA_DATASET")

print(f"Saved dataset to {DB_NAME}.{SCHEMA_NAME}.VQA_DATASET")
session.table(f"{DB_NAME}.{SCHEMA_NAME}.VQA_DATASET").show(5)

## Step 6: Format Input for Batch Inference

In [ ]:
def create_chat_message(row):
    return json.dumps([
        {
            "role": "user",
            "content": [
                {"type": "text", "text": f"{row['QUESTION']} Answer briefly in a few words."},
                {"type": "image_url", "image_url": {"url": row['IMAGE_PATH']}}
            ]
        }
    ])

df_vqa['MESSAGES'] = df_vqa.apply(create_chat_message, axis=1)

input_df = session.create_dataframe(df_vqa[['ID', 'MESSAGES', 'QUESTION', 'ANSWER', 'QUESTION_TYPE']])
input_df.write.mode("overwrite").save_as_table(f"{DB_NAME}.{SCHEMA_NAME}.INFERENCE_INPUT")

print(f"Prepared {input_df.count()} samples for inference")
print("\nSample message format:")
print(json.dumps(json.loads(df_vqa.iloc[0]['MESSAGES']), indent=2))

## Done!

Now proceed to `02_batch_inference.ipynb` to run the model.